# Optimizer scenario pack

This notebook runs 20 curated roster variations against the current lineup optimizer. It is designed to exercise repeated daily runs and edge cases without hitting Yahoo or MLB APIs.

In [8]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

from lineup import optimize_lineup, render_plan, render_roster
from scenario_fixtures import SCENARIO_PROJECTIONS, expected_signature, scenario_catalog


In [10]:
scenarios = scenario_catalog()
len(scenarios)


27

In [11]:
results = []
plans = {}

for scenario in scenarios:
    plan = optimize_lineup(scenario.roster, projections=SCENARIO_PROJECTIONS)
    plans[scenario.name] = plan
    actual_moves = expected_signature(plan)
    actual_warnings = tuple(plan.warnings)
    results.append({
        'name': scenario.name,
        'description': scenario.description,
        'expected_move_count': len(scenario.expected_moves),
        'actual_move_count': len(actual_moves),
        'moves_match': actual_moves == scenario.expected_moves,
        'warnings_match': actual_warnings == scenario.expected_warnings,
        'expected_moves': '\n'.join(scenario.expected_moves),
        'actual_moves': '\n'.join(actual_moves),
        'expected_warnings': '\n'.join(scenario.expected_warnings),
        'actual_warnings': '\n'.join(actual_warnings),
    })

results_df = pd.DataFrame(results)
results_df[['name', 'expected_move_count', 'actual_move_count', 'moves_match', 'warnings_match']]


,name,expected_move_count,actual_move_count,moves_match,warnings_match
0,baseline,6,6,True,True
1,cf_no_game_jo_adell,8,8,True,True
2,no_game_can_use_lineup_pending_bench,2,2,True,True
3,no_game_vs_lineup_pending_only,2,2,True,True
4,of_not_starting_jo_adell,8,8,True,True
5,o_rank_star_upgrade,2,2,True,True
6,catcher_warning_only,2,2,True,True
7,prefer_projection_for_2b,6,6,True,True
8,locked_active_skipped,4,4,True,True
9,locked_bench_unavailable,4,4,True,True


In [12]:
results_df[~(results_df['moves_match'] & results_df['warnings_match'])][[
    'name', 'description', 'expected_moves', 'actual_moves', 'expected_warnings', 'actual_warnings'
]]


,name,description,expected_moves,actual_moves,expected_warnings,actual_warnings


In [13]:
scenario_name = 'o_rank_star_upgrade'
scenario = next(item for item in scenarios if item.name == scenario_name)
print(scenario.description)
print()
print(render_roster(scenario.roster))
print()
print(render_plan(plans[scenario_name]))


A higher-ranked star on the bench should replace a lower-ranked starter even when both are starting.

Team: deGrom Reapers
Date: 2025-09-28
Slots: C x1, 1B x1, 2B x1, 3B x1, SS x1, IF x1, LF x1, CF x1, RF x1, OF x1, Util x1, SP x3, RP x3, P x3
Roster:
- 1B   Brandon Lowe (1B,2B,IF,Util) [player unmapped]
- 2B   Jordan Westburg (2B,3B) [starting]
- 3B   Mark Vientos (3B,IF,Util) [starting]
- C    William Contreras (C,Util) [starting]
- CF   Wyatt Langford (LF,CF,OF,Util) [player unmapped]
- IF   Gleyber Torres (2B,IF,Util) [starting]
- LF   George Springer (LF,CF,RF,OF,Util) [starting]
- OF   Brenton Doyle (CF,OF,Util) [starting]
- P    Brandon Woodruff (SP,P) [player unmapped]
- P    Daniel Palencia (RP,P) [reliever]
- P    Jonah Tong (SP,P,NA) [starting]
- RF   Jackson Chourio (LF,CF,RF,OF,Util) [starting]
- RP   Dennis Santana (RP,P) [reliever]
- RP   Devin Williams (RP,P) [reliever]
- SP   Hunter Brown (SP,P) [starting]
- SP   Jacob deGrom (SP,P) [starting]
- SP   Parker Messick (SP